# BurnSense — Analisis Perbandingan Fuzzy Mamdani dan Sugeno
## Prediksi Tingkat Burnout Karyawan

**Dataset:** [Are Your Employees Burning Out?](https://www.kaggle.com/datasets/blurredmachine/are-your-employees-burning-out)  
**Metode:** Fuzzy Mamdani & Sugeno (from scratch)  
**Evaluasi:** MAE, MSE, RMSE  


## 1. Import Library & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time, sys, os

sys.path.insert(0, os.path.abspath('..'))

from fuzzy.membership import fuzzify_all, get_mf_curves, SUGENO_CONSTANTS, mf_output
from fuzzy.mamdani import mamdani_predict, mamdani_evaluate_batch, RULES, get_burnout_level
from fuzzy.sugeno import sugeno_predict, sugeno_evaluate_batch

PINK = {
    'primary':   '#D4537E', 'secondary': '#ED93B1',
    'light':     '#F4C0D1', 'lighter':   '#FBEAF0',
    'dark':      '#72243E', 'mid':       '#993556', 'muted': '#C47A95',
}
print("Library berhasil diimport!")

## 2. Load & Preprocessing Dataset

In [ ]:
df = pd.read_csv('../data/train.csv')
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Preprocessing
df_clean = df.copy()
df_clean = df_clean.drop(columns=['Employee ID', 'Date of Joining', 'Gender'], errors='ignore')
df_clean['WFH_encoded']     = df_clean['WFH Setup Available'].map({'Yes': 1, 'No': 0})
df_clean['Company_encoded'] = df_clean['Company Type'].map({'Product': 1, 'Service': 0})
df_clean['Mental Fatigue Score'] = df_clean['Mental Fatigue Score'].fillna(df_clean['Mental Fatigue Score'].median())
df_clean['Resource Allocation']  = df_clean['Resource Allocation'].fillna(df_clean['Resource Allocation'].median())
df_clean['Burn Rate']            = df_clean['Burn Rate'].fillna(df_clean['Burn Rate'].median())
print("Missing values setelah preprocessing:")
print(df_clean.isnull().sum())
df_clean.describe()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.patch.set_facecolor('white')
fig.suptitle('EDA — Burnout Dataset', color=PINK['dark'], fontsize=14, fontweight='bold')

axes[0,0].hist(df_clean['Burn Rate'], bins=30, color=PINK['primary'], alpha=0.85, edgecolor='white')
axes[0,0].set_title('Distribusi Burn Rate', color=PINK['dark'])

axes[0,1].hist(df_clean['Mental Fatigue Score'], bins=25, color=PINK['secondary'], alpha=0.85, edgecolor='white')
axes[0,1].set_title('Distribusi Mental Fatigue', color=PINK['dark'])

axes[0,2].hist(df_clean['Resource Allocation'], bins=20, color=PINK['mid'], alpha=0.85, edgecolor='white')
axes[0,2].set_title('Distribusi Resource Allocation', color=PINK['dark'])

ct = df_clean['Company Type'].value_counts()
axes[1,0].bar(ct.index, ct.values, color=[PINK['primary'], PINK['secondary']], edgecolor='white')
axes[1,0].set_title('Company Type', color=PINK['dark'])

wfh = df_clean['WFH Setup Available'].value_counts()
axes[1,1].bar(wfh.index, wfh.values, color=[PINK['light'], PINK['primary']], edgecolor='white')
axes[1,1].set_title('WFH Setup Available', color=PINK['dark'])

axes[1,2].scatter(df_clean['Mental Fatigue Score'], df_clean['Burn Rate'],
                  alpha=0.2, color=PINK['primary'], s=5)
axes[1,2].set_title('Fatigue vs Burn Rate', color=PINK['dark'])

for ax in axes.flat:
    ax.tick_params(colors=PINK['muted'])
    for sp in ax.spines.values(): sp.set_color(PINK['light'])

plt.tight_layout()
plt.show()
print(f"Korelasi Fatigue vs Burn Rate: {df_clean['Mental Fatigue Score'].corr(df_clean['Burn Rate']):.3f}")

## 4. Membership Function

In [ ]:
curves = get_mf_curves()
labels_map = {
    'fatigue':     (['LOW', 'MEDIUM', 'HIGH'], 'Mental Fatigue Score (0-10)'),
    'resource':    (['LOW', 'MEDIUM', 'HIGH'], 'Resource Allocation (1-10)'),
    'designation': (['JUNIOR', 'MID', 'SENIOR'], 'Designation (0-5)'),
    'output':      (['LOW', 'MODERATE', 'HIGH', 'SEVERE'], 'Output Burn Rate (0-1)'),
}
colors_mf = [PINK['primary'], PINK['secondary'], PINK['mid'], PINK['light']]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.patch.set_facecolor('white')
fig.suptitle('Membership Functions — BurnoutSense AI', color=PINK['dark'], fontsize=13, fontweight='bold')

for idx, (key, (lbls, xlabel)) in enumerate(labels_map.items()):
    ax = axes[idx//2][idx%2]
    c  = curves[key]
    for j, lbl in enumerate(lbls):
        ax.plot(c['x'], c[lbl], color=colors_mf[j], linewidth=2.5, label=lbl)
        ax.fill_between(c['x'], c[lbl], alpha=0.08, color=colors_mf[j])
    ax.set_title(xlabel, color=PINK['dark'], fontsize=11)
    ax.set_ylabel('µ', color=PINK['muted'])
    ax.legend(fontsize=9)
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', color=PINK['lighter'], linewidth=0.5)
    ax.tick_params(colors=PINK['muted'])
    for sp in ax.spines.values(): sp.set_color(PINK['light'])

plt.tight_layout()
plt.show()

## 5. Rule Base (20 Rules)

In [ ]:
print("RULE BASE:")
for i, rule in enumerate(RULES):
    conds = ' AND '.join([f"{v}={l}" for v, l in rule['conditions']])
    print(f"R{i+1:02d}: IF {conds} THEN burnout={rule['output']}")

## 6. Demo Fuzzifikasi

In [ ]:
fatigue=7.0; resource=8.0; designation=4.0; wfh=0; company=1
fuzz = fuzzify_all(fatigue, resource, designation, wfh, company)
print("HASIL FUZZIFIKASI:")
for var, vals in fuzz.items():
    print(f"  {var:<12}", end="")
    for lbl, mu in vals.items():
        print(f"  {lbl}: {mu:.3f}", end="")
    print()

## 7. Fuzzy Mamdani

In [ ]:
result_m = mamdani_predict(fatigue, resource, designation, wfh, company)
print(f"Burn Rate : {result_m['burn_rate']}")
print(f"Level     : {result_m['level']}")
print(f"Runtime   : {result_m['runtime']}s")
print(f"Rule aktif: {len(result_m['fired_rules'])}")
for r in result_m['fired_rules']:
    conds = ' AND '.join([f"{v}={l}" for v, l in r['conditions']])
    print(f"  R{r['rule_idx']:02d}: {conds} -> {r['output']} (strength={r['firing_strength']:.3f})")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('white')
ax.fill_between(result_m['x_values'], result_m['aggregated'], alpha=0.3, color=PINK['primary'])
ax.plot(result_m['x_values'], result_m['aggregated'], color=PINK['primary'], linewidth=2)
ax.axvline(result_m['burn_rate'], color=PINK['dark'], linestyle='--', linewidth=2,
           label=f"Centroid = {result_m['burn_rate']:.3f}")
ax.set_title('Mamdani — Kurva Agregasi & Centroid', color=PINK['dark'], fontsize=12)
ax.set_xlabel('Burn Rate', color=PINK['muted'])
ax.set_ylabel('µ', color=PINK['muted'])
ax.legend()
for sp in ax.spines.values(): sp.set_color(PINK['light'])
plt.tight_layout(); plt.show()

## 8. Fuzzy Sugeno

In [ ]:
result_s = sugeno_predict(fatigue, resource, designation, wfh, company)
print(f"Burn Rate : {result_s['burn_rate']}")
print(f"Level     : {result_s['level']}")
print(f"Runtime   : {result_s['runtime']}s")
print()
print("Kontribusi setiap rule (weighted average):")
for r in result_s['fired_rules']:
    print(f"  {r['output']}: w={r['firing_strength']:.3f} x z={r['output_value']}")
print(f"\nOutput = weighted average = {result_s['burn_rate']}")

## 9. Perbandingan Mamdani vs Sugeno

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('white')
fig.suptitle('Perbandingan Mamdani vs Sugeno', color=PINK['dark'], fontsize=13, fontweight='bold')

methods = ['Mamdani', 'Sugeno']
values  = [result_m['burn_rate'], result_s['burn_rate']]
axes[0].bar(methods, values, color=[PINK['primary'], PINK['secondary']], alpha=0.85, edgecolor='white', width=0.4)
axes[0].set_title('Output Burn Rate', color=PINK['dark'])
axes[0].set_ylim(0, 1)
for i, v in enumerate(values):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', color=PINK['dark'])
for sp in axes[0].spines.values(): sp.set_color(PINK['light'])

rt_vals = [result_m['runtime'], result_s['runtime']]
axes[1].bar(methods, rt_vals, color=[PINK['primary'], PINK['secondary']], alpha=0.85, edgecolor='white', width=0.4)
axes[1].set_title('Runtime (s)', color=PINK['dark'])
for i, v in enumerate(rt_vals):
    axes[1].text(i, v * 1.05, f'{v:.4f}s', ha='center', color=PINK['dark'])
for sp in axes[1].spines.values(): sp.set_color(PINK['light'])

plt.tight_layout(); plt.show()

## 10. Evaluasi Batch — MAE, MSE, RMSE

In [ ]:
SAMPLE = 2000
print(f"Evaluasi pada {SAMPLE} data...")

t0 = time.time()
actual_m, pred_m = mamdani_evaluate_batch(df_clean, SAMPLE)
rt_m = time.time() - t0
mae_m  = np.mean(np.abs(actual_m - pred_m))
mse_m  = np.mean((actual_m - pred_m)**2)
rmse_m = np.sqrt(mse_m)

t0 = time.time()
actual_s, pred_s = sugeno_evaluate_batch(df_clean, SAMPLE)
rt_s = time.time() - t0
mae_s  = np.mean(np.abs(actual_s - pred_s))
mse_s  = np.mean((actual_s - pred_s)**2)
rmse_s = np.sqrt(mse_s)

print(f"{'Metrik':<12} {'Mamdani':>12} {'Sugeno':>12}")
print("-"*40)
print(f"{'MAE':<12} {mae_m:>12.4f} {mae_s:>12.4f}")
print(f"{'MSE':<12} {mse_m:>12.4f} {mse_s:>12.4f}")
print(f"{'RMSE':<12} {rmse_m:>12.4f} {rmse_s:>12.4f}")
print(f"{'Runtime(s)':<12} {rt_m:>12.2f} {rt_s:>12.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.patch.set_facecolor('white')
fig.suptitle('Evaluasi Performa: Mamdani vs Sugeno', color=PINK['dark'], fontsize=13, fontweight='bold')
metrics = [('MAE', [mae_m, mae_s]), ('RMSE', [rmse_m, rmse_s]), ('Runtime (s)', [rt_m, rt_s])]
for i, (title, vals) in enumerate(metrics):
    bars = axes[i].bar(['Mamdani','Sugeno'], vals, color=[PINK['primary'], PINK['secondary']], alpha=0.85, edgecolor='white', width=0.4)
    axes[i].set_title(title, color=PINK['dark'])
    axes[i].tick_params(colors=PINK['muted'])
    for sp in axes[i].spines.values(): sp.set_color(PINK['light'])
    for bar in bars:
        h = bar.get_height()
        axes[i].text(bar.get_x()+bar.get_width()/2, h*1.02, f'{h:.4f}', ha='center', fontsize=9, color=PINK['dark'])
plt.tight_layout(); plt.show()

## 11. Bonus: Fuzzy + Machine Learning

In [ ]:
from ml.random_forest import generate_fuzzy_features, train_and_evaluate
df_fuzzy = generate_fuzzy_features(df_clean, sample_size=2000)
results  = train_and_evaluate(df_fuzzy)

print(f"{'Metrik':<12} {'RF raw':>12} {'RF+Fuzzy':>12} {'Improve':>10}")
print("-"*50)
for m in ['mae', 'mse', 'rmse']:
    v1 = results['rf_raw'][m]
    v2 = results['rf_fuzzy'][m]
    imp = (v1 - v2) / v1 * 100
    print(f"{m.upper():<12} {v1:>12.4f} {v2:>12.4f} {imp:>9.1f}%")
print("\nFuzzy features terbukti meningkatkan akurasi RF!")

## 12. Kesimpulan

### Perbandingan Mamdani vs Sugeno

| Aspek | Mamdani | Sugeno |
|-------|---------|--------|
| Defuzzifikasi | Centroid (integrasi area) | Weighted Average |
| Akurasi | Lebih baik (MAE lebih rendah) | Sedikit lebih tinggi error |
| Kecepatan | Lebih lambat | Jauh lebih cepat |
| Interpretabilitas | Sangat tinggi | Sedang |
| Cocok untuk | Sistem eksplanatif | Sistem real-time & ML |

### Kelebihan & Kekurangan
- **Mamdani**: (+) Akurat, ekspresif, mudah diinterpretasi. (-) Lambat untuk data besar.
- **Sugeno**: (+) Cepat, efisien, mudah diintegrasikan. (-) Kurang natural untuk interpretasi.

### Bonus ML
- Output fuzzy (Mamdani + Sugeno) terbukti sebagai **fitur informatif** untuk Random Forest.
- Model RF + Fuzzy Features menghasilkan error lebih rendah dibanding RF tanpa fuzzy.

### Future Work
- Eksplorasi **ANFIS** (Adaptive Neuro-Fuzzy Inference System) — membership function yang dipelajari dari data.
